# Kaggle Data Science Survey Trends (2017–2021)

A reproducible analysis of participation, demographics, roles, and programming-language usage across five annual Kaggle surveys.

## Questions

1. How did survey participation change?
2. Which countries and roles were most represented?
3. Which programming languages were used most regularly?
4. How did gender representation change?

Results describe survey respondents, not the entire data-science workforce.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
RAW = ROOT / 'kaggle_survey_2017_2021.zip'
raw = pd.read_csv(RAW, low_memory=False)
questions = raw.iloc[0].copy()
df = raw.iloc[1:].rename(columns={'-': 'Year', 'Time from Start to Finish (seconds)': 'DurationSeconds'}).copy()
df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int64')
df = df[df['Year'].isin([2017, 2018, 2019, 2020, 2021])].reset_index(drop=True)
df.shape

## Data quality and harmonization

- Remove the embedded question-description row from respondent records.
- Retain identical answer profiles because the merged file has no respondent ID.
- Standardize gender labels that changed between surveys.
- Repair known spreadsheet date conversions in the team-size field.
- Preserve missing answers; non-response is not treated as zero.

In [ ]:
def fix_mojibake(value):
    if not isinstance(value, str) or 'â' not in value:
        return value
    try:
        return value.encode('latin1').decode('utf-8')
    except (UnicodeEncodeError, UnicodeDecodeError):
        return value

for col in ['Q1','Q2','Q3','Q4','Q5','Q6','Q8','Q15','Q22','Q23','Q25','Q41']:
    df[col] = df[col].map(fix_mojibake)

df['Gender'] = df['Q2'].replace({'Male':'Man','Female':'Woman','Prefer to self-describe':'Self-described'})
df['CodingExperience'] = df['Q6'].replace({'< 1 years':'<1 year','< 1 year':'<1 year'})
df['DataScienceTeamSize'] = df['Q22'].replace({'2-Jan':'1-2','4-Mar':'3-4','9-May':'5-9','14-Oct':'10-14'})

In [ ]:
quality = pd.Series({
    'respondent_rows': len(df),
    'years': f"{df['Year'].min()}–{df['Year'].max()}",
    'identical_answer_profiles_retained': int(df.duplicated().sum()),
    'countries': int(df['Q3'].nunique()),
})
quality

## Participation

In [ ]:
annual = df.groupby('Year').size().rename('Respondents').to_frame()
annual['GrowthVsPriorYear'] = annual['Respondents'].pct_change()
annual

In [ ]:
ax = annual['Respondents'].plot(marker='o', figsize=(9,4.5), color='#4C78A8', linewidth=2.5)
ax.set(title='Survey respondents by year', xlabel='Survey year', ylabel='Respondents')
ax.grid(axis='y', alpha=.25)
plt.show()

## Geography

In [ ]:
top_countries_2021 = df.loc[df['Year'].eq(2021), 'Q3'].value_counts().drop(labels=['Other'], errors='ignore').head(12)
top_countries_2021

In [ ]:
ax = top_countries_2021.sort_values().plot.barh(figsize=(9,5.5), color='#2A9D8F')
ax.set(title='Top respondent countries in 2021', xlabel='Respondents', ylabel='')
ax.grid(axis='x', alpha=.25)
plt.show()

## Programming languages

The merged file does not contain the regular-use multi-select fields for 2017, so this comparison begins in 2018.

In [ ]:
language_cols = [c for c in df.columns if c.startswith('Q7_')]
rows = []
for year, group in df.groupby('Year'):
    for col in language_cols:
        for language, count in group[col].dropna().astype(str).str.strip().value_counts().items():
            if language:
                rows.append({'Year':year, 'Language':language, 'Selections':count, 'Respondents':len(group)})
languages = pd.DataFrame(rows).groupby(['Year','Language','Respondents'], as_index=False)['Selections'].sum()
languages['Share'] = languages['Selections'] / languages['Respondents']
languages.head()

In [ ]:
top_languages = languages.groupby('Language')['Selections'].sum().nlargest(6).index
language_trend = languages[languages['Language'].isin(top_languages)].pivot(index='Year', columns='Language', values='Share')
ax = (100*language_trend).plot(marker='o', figsize=(10,5))
ax.set(title='Regularly used programming languages', xlabel='Survey year', ylabel='Share of all respondents (%)')
ax.grid(axis='y', alpha=.25)
plt.show()

## Gender representation

In [ ]:
gender = df.groupby(['Year','Gender']).size().rename('Count').reset_index()
gender['Share'] = gender['Count'] / gender.groupby('Year')['Count'].transform('sum')
gender[gender['Gender'].eq('Woman')][['Year','Count','Share']]

In [ ]:
women = gender[gender['Gender'].eq('Woman')].set_index('Year')['Share'] * 100
ax = women.plot(marker='o', figsize=(9,4.5), color='#F4A261', linewidth=2.5)
ax.set(title='Women among survey respondents', xlabel='Survey year', ylabel='Share of all respondents (%)')
ax.grid(axis='y', alpha=.25)
plt.show()

## Roles in 2021

In [ ]:
role_mix_2021 = df.loc[df['Year'].eq(2021), 'Q5'].value_counts(normalize=True).head(10).mul(100)
role_mix_2021.round(1)

In [ ]:
ax = role_mix_2021.sort_values().plot.barh(figsize=(9,5.5), color='#4C78A8')
ax.set(title='Largest respondent roles in 2021', xlabel='Share of respondents (%)', ylabel='')
ax.grid(axis='x', alpha=.25)
plt.show()

## Conclusions

- Participation reached its highest level in 2021 after a dip in 2019.
- India and the United States accounted for the largest respondent counts in 2021.
- Python was the most commonly selected regular-use language throughout the period.
- Women remained materially underrepresented in the respondent pool.
- Students were the largest reported role group in 2021.

These are descriptive survey findings. Changes can reflect both ecosystem shifts and changes in sampling, questionnaire wording, or response patterns.